## Assignment: Image recognition
- Alumno 1: Juan García Santos
- Alumno 2: Guiilermo Alonso Rello 
- Alumno 3: Aritz Bermejo Canal

The goals of the assignment are:
* Develop proficiency in using Tensorflow/Keras for training Neural Nets (NNs).
* Put into practice the acquired knowledge to optimize the parameters and architecture of a feedforward Neural Net (ffNN), in the context of an image recognition problem.
* Put into practice NNs specially conceived for analysing images. Design and optimize the parameters of a Convolutional Neural Net (CNN) to deal with previous task.
* Train popular architectures from scratch (e.g., GoogLeNet, VGG, ResNet, ...), and compare the results with the ones provided by their pre-trained versions using transfer learning.

Follow the link below to download the classification data set  “xview_recognition”: [https://drive.upm.es/s/4oNHlRFEd71HXp4](https://drive.upm.es/s/4oNHlRFEd71HXp4)

In [ ]:
import uuid
import numpy as np

class GenericObject:
    """
    Generic object data.
    """
    def __init__(self):
        self.id = uuid.uuid4()
        self.bb = (-1, -1, -1, -1)
        self.category= -1
        self.score = -1

class GenericImage:
    """
    Generic image data.
    """
    def __init__(self, filename):
        self.filename = filename
        self.tile = np.array([-1, -1, -1, -1])  # (pt_x, pt_y, pt_x+width, pt_y+height)
        self.objects = list([])

    def add_object(self, obj: GenericObject):
        self.objects.append(obj)

In [ ]:
categories = {0: 'Cargo plane', 1: 'Helicopter', 2: 'Small car', 3: 'Bus', 4: 'Truck', 5: 'Motorboat', 6: 'Fishing vessel', 7: 'Dump truck', 8: 'Excavator', 9: 'Building', 10: 'Storage tank', 11: 'Shipping container'}

categories_inv = {}
for i in categories.keys():
    categories_inv[categories[i]] = i

Augmentation = True

In [ ]:
import warnings
import rasterio
from tensorflow.keras.applications.resnet50 import preprocess_input as preprocess_resnet
from tensorflow.keras.applications.resnet_v2 import preprocess_input as preprocess_resnetV2

# Devuelve la matriz 3D con los bits de una imagen dado el nombre del archivo
def load_geoimage(filename):
    if Augmentation:
        filename = '../input/xviewrecognitionaug/' + filename
    else:
        filename = '../input/xviewrecognition/' + filename
    warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
    src_raster = rasterio.open(filename, 'r')
    # RasterIO to OpenCV (see inconsistencies between libjpeg and libjpeg-turbo)
    input_type = src_raster.profile['dtype']
    input_channels = src_raster.count # 3 canales (RGB)
    img = np.zeros((src_raster.height, src_raster.width, src_raster.count), dtype=input_type)
    for band in range(input_channels): # Rellenar toda la matriz 2D del canal k de la matriz
        img[:, :, band] = src_raster.read(band+1)
    return img

# Genera grupos de imágenes del tamaño del batch durante todo el proceso de entrenamiento
def generator_images(objs, batch_size, batch_size_aug, train_dataset_size, 
                     preprocess_function, do_shuffle=False):
    while True:
        if do_shuffle:
            np.random.shuffle(objs)
        groups = [objs[i:i+batch_size] for i in range(0, len(objs), batch_size)]
        for group in groups: # 1 epoch
            images, labels = [], [] # Matrices de imagen y one-hot encoding de cada una del grupo
            for (filename, obj) in group: # 1 mini-batch
                # Load image
                img = load_geoimage(filename)
                images.append(preprocess_function(img)) # Bits de la imagen normalizados
                probabilities = np.zeros(len(categories)) # Vector para one-hot enconding
                probabilities[list(categories.values()).index(obj.category)] = 1
                labels.append(probabilities)
            images = np.array(images).astype(np.float32)
            labels = np.array(labels).astype(np.float32)
            yield images, labels # Devolver todo el grupo
        if(batch_size < train_dataset_size/10): # Al final de la epoch
            batch_size = batch_size * batch_size_aug
            print(batch_size)
            

def get_preprocessing(base_model):
    preprocess_function = None
    
    if base_model.name == "resnet50":
        preprocess_function = preprocess_resnet
        
    if base_model.name == "resnet152v2":
        preprocess_function = preprocess_resnetV2
        
    print(base_model.name)
        
    return preprocess_function

In [ ]:
#### Funciones de test

import matplotlib.pyplot as plt

def test_net(model, test_anns, preprocess_function):
    y_true, y_pred = [], []
    for ann in test_anns:
        # Load image
        image = preprocess_function(load_geoimage(ann.filename))
        for obj_pred in ann.objects:
            # Generate prediction
            warped_image = np.expand_dims(image, 0)
            predictions = model.predict(warped_image, verbose = 0)
            # Save prediction
            pred_category = list(categories.values())[np.argmax(predictions)]
            pred_score = np.max(predictions)
            y_true.append(obj_pred.category)
            y_pred.append(pred_category)
    return y_true, y_pred

def draw_confusion_matrix(cm, categories):
    # Draw confusion matrix
    fig = plt.figure(figsize=[6.4*pow(len(categories), 0.5), 4.8*pow(len(categories), 0.5)])
    ax = fig.add_subplot(111)
    cm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], np.finfo(np.float64).eps)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.get_cmap('Blues'))
    ax.figure.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), xticklabels=list(categories.values()), yticklabels=list(categories.values()), ylabel='Annotation', xlabel='Prediction')
    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    # Loop over data dimensions and create text annotations
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], '.2f'), ha="center", va="center", color="white" if cm[i, j] > thresh else "black", fontsize=int(20-pow(len(categories), 0.5)))
    fig.tight_layout()
    plt.show(fig)

# Imprimir resultados a partir de la matriz de confusión
def print_results(cm):
    # Compute the accuracy
    correct_samples_class = np.diag(cm).astype(float)
    total_samples_class = np.sum(cm, axis=1).astype(float)
    total_predicts_class = np.sum(cm, axis=0).astype(float)
    print('Mean Accuracy: %.3f%%' % (np.sum(correct_samples_class) / np.sum(total_samples_class) * 100))
    acc = correct_samples_class / np.maximum(total_samples_class, np.finfo(np.float64).eps)
    print('Mean Recall: %.3f%%' % (acc.mean() * 100))
    acc = correct_samples_class / np.maximum(total_predicts_class, np.finfo(np.float64).eps)
    print('Mean Precision: %.3f%%' % (acc.mean() * 100))
    for idx in range(len(categories)):
        # True/False Positives (TP/FP) refer to the number of predicted positives that were correct/incorrect.
        # True/False Negatives (TN/FN) refer to the number of predicted negatives that were correct/incorrect.
        tp = cm[idx, idx]
        fp = sum(cm[:, idx]) - tp
        fn = sum(cm[idx, :]) - tp
        tn = sum(np.delete(sum(cm) - cm[idx, :], idx))
        # True Positive Rate: proportion of real positive cases that were correctly predicted as positive.
        recall = tp / np.maximum(tp+fn, np.finfo(np.float64).eps)
        # Precision: proportion of predicted positive cases that were truly real positives.
        precision = tp / np.maximum(tp+fp, np.finfo(np.float64).eps)
        # True Negative Rate: proportion of real negative cases that were correctly predicted as negative.
        specificity = tn / np.maximum(tn+fp, np.finfo(np.float64).eps)
        # Dice coefficient refers to two times the intersection of two sets divided by the sum of their areas.
        # Dice = 2 |A∩B| / (|A|+|B|) = 2 TP / (2 TP + FP + FN)
        f1_score = 2 * ((precision * recall) / np.maximum(precision+recall, np.finfo(np.float64).eps))
        print('> %s: Recall: %.3f%% Precision: %.3f%% Specificity: %.3f%% Dice: %.3f%%' % (list(categories.values())[idx], recall*100, precision*100, specificity*100, f1_score*100))

#### Training
Design and train a ffNN to deal with the “xview_recognition” classification task.

In [ ]:
#### Funciones para cargar BD
import json
from sklearn.utils.class_weight import compute_class_weight

# Load JSON and then use it to get objects
def import_database(json_file):
    # Load database JSON.
    with open(json_file) as ifs:
        json_data = json.load(ifs)
    ifs.close()

    counts = dict.fromkeys(categories.values(), 0)
    anns = []
    all_labels = []
    # Crear un GenericImage por cada archivo con su GenericObject correspondiente categorizado dentro
    for json_img, json_ann in zip(json_data['images'].values(), json_data['annotations'].values()):
        image = GenericImage(json_img['filename'])
        image.tile = np.array([0, 0, json_img['width'], json_img['height']])
        obj = GenericObject()
        obj.bb = (int(json_ann['bbox'][0]), int(json_ann['bbox'][1]), int(json_ann['bbox'][2]), int(json_ann['bbox'][3]))
        obj.category = json_ann['category_id']
        all_labels.append(obj.category)
        # Resampling strategy to reduce training time
        counts[obj.category] += 1
        image.add_object(obj)
        anns.append(image)
    print(counts)
    labels_unique = np.array(list(categories.keys()))
    
    for i in range(len(all_labels)):
        all_labels[i] = categories_inv[all_labels[i]]
    all_labels = np.array(all_labels,dtype=int)
    
    class_weights = compute_class_weight('balanced',
                                    classes = labels_unique,
                                     y = all_labels)

    class_weight_dict = {label: weight 
                     for label,weight in zip(labels_unique, class_weights)}

    return anns, class_weight_dict

In [ ]:
# Cargamos las BD de train y test
import os
print(os.listdir('/kaggle/input/'))
if Augmentation:
    json_train_file = '../input/xviewrecognitionaug/xview_ann_train.json'
    json_test_file = '../input/xviewrecognitionaug/xview_ann_test.json'
else:
    json_train_file = '../input/xviewrecognition/xview_ann_train.json'
    json_test_file = '../input/xviewrecognition/xview_ann_test.json'
print('Train')
anns_train, class_weight_dict = import_database(json_train_file)
print('Test')
anns_test, _ = import_database(json_test_file)
class_weight_dict

In [ ]:
from sklearn.model_selection import train_test_split

anns_train, anns_valid = train_test_split(anns_train, test_size=0.1, random_state=1, shuffle=True)
N_train = len(anns_train)

In [ ]:
def get_lr_schedule(N_steps, percentage_warmup, initial_learning_rate, target_warm_learning_rate, final_learning_rate):
    warmup_steps = math.floor(percentage_warmup * N_steps) # Step = minibatch
    decay_steps = math.floor((1 - percentage_warmup) * N_steps) # Pasos de decay coseno
    l_rate_decay = CosineDecay(
        initial_learning_rate, decay_steps, alpha=final_learning_rate, 
        #warmup_target=target_warm_learning_rate, warmup_steps=warmup_steps
    )
    print('Warmup', warmup_steps, 'Decay', decay_steps)
    return l_rate_decay

In [ ]:
from tensorflow.keras.optimizers.schedules import CosineDecay

from tensorflow_addons.metrics import FBetaScore
import math
import numpy as np
from tensorflow import random

random.set_seed(120)

percentage_warmup = 0
initial_learning_rate = 0.1 # Antes del warmup
target_warm_learning_rate = 0.1 # Después del warmup
final_learning_rate = 1e-6 # el learning rate mínimo

cosine_decay_params = [percentage_warmup, initial_learning_rate, target_warm_learning_rate, final_learning_rate]

num_models = 1
b_size = [64]*num_models
N_epochs = [10]
l_rate = [cosine_decay_params]*num_models
#l_rate = [0.01]*num_models
optimizer = ['SGD']*num_models

# Allowed kwargs are {'global_clipnorm', 'decay', 'lr', 'clipnorm', 'clipvalue'}.
opt_hparams = []

#opt_hparams = [{'weight_decay':0.004,'beta_1':0.9,'beta_2':0.999,'epsilon':1e-07}] * num_models

opt_hparams = [{'momentum':0.9, 'nesterov': False}] * num_models

neurons_hidden_layers = [[256, 256]] * num_models
activ_fun =  ['relu'] * num_models
unfreeze = [-1] * num_models # Si usamos multi-step, no poner unfreeze a 0
multi_step = [False]
inference = [True] * num_models # Si se congela o no batchnorm

#class_weight = [class_weight_dict] * num_models
class_weight = [None] * num_models

patience_lr = [1] * num_models
patience_es = [2] * num_models    
    
f1 = FBetaScore(num_classes = 12,average = 'weighted')
#metric = [f1]*num_models
metric = [['accuracy']]*num_models

In [ ]:
import math
import datetime
from tensorflow.keras.callbacks import TerminateOnNaN, EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
def get_callbacks(patience_lr, patience_es):
    # Callbacks
    current_time = datetime.datetime.now()
    time_string = current_time.strftime("%Y-%m-%d_%H-%M-%S") # Replace colons with underscores
    
    model_checkpoint = ModelCheckpoint(f"{time_string}.tf", monitor='val_accuracy', verbose=1, save_best_only=True)
    
    early_stop = EarlyStopping('val_accuracy', patience=patience_es, verbose=1)
    terminate = TerminateOnNaN()    
    tensorboard = TensorBoard(log_dir=f"./logs/{time_string}")
    
    if patience_lr > 0:
        reduce_lr = ReduceLROnPlateau('val_accuracy', factor=0.1, patience=patience_lr, verbose=1) # Reducir el learning rate si se estanca
        callbacks = [model_checkpoint, reduce_lr, early_stop, terminate, tensorboard]
    else:
        callbacks = [model_checkpoint, early_stop, terminate, tensorboard]
    
    
    return callbacks, time_string

In [ ]:
def print_params(date_time, l_rate, b_size, bs_aug_factor, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun, patience_lr, patience_es, unfreeze):
    print('Model parameters')
    print('Datetime', date_time)
    if isinstance(l_rate, float):
        print('Learning rate (fijo)', l_rate)
    else:
        print('Cosine decay')
        print('Learning rate antes del warmup', l_rate.initial_learning_rate)
        #print('Numero de steps en warnup', l_rate.warmup_steps)
        #print('Learning rate después del warmup', l_rate.warmup_target) 
        print('Numero de decay steps coseno', l_rate.decay_steps)
        print('Minimo learning rate con decay coseno', l_rate.alpha)
    print('Batch size', b_size)
    print('Epochs', epochs)
    print('Optimizer', optimizer)
    print('Optimizer hiperparams', opt_hparams)
    print('Capas ocultas', neurons_hidden_layers)
    print('Función de activación', activ_fun)
    print('Capas descongeladas', unfreeze)
    print('Patience learning rate on plateau', patience_lr)
    print('Patience early stopping', patience_es)

In [ ]:

from tensorflow.keras import Input
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, BatchNormalization, Conv2D, MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.regularizers import L2
from tensorflow.keras.optimizers import Adam, AdamW, SGD
#from tensorflow.keras.optimizers.legacy import SGD

def unfreeze_model (base_model, unfreeze: int, inference):
    if unfreeze == 0:
        return 
    elif unfreeze == - 1:
        base_model.trainable = True
        return
    
    if base_model.name == 'resnet50':
        unfreeze_residual_blocks(base_model, unfreeze, inference)
        
    elif base_model.name == 'Pon aqui el nombre que salga para el modelo efficient net escogido':
        unfreeze_efficient_net(base_model, unfreeze)
    
def unfreeze_efficient_net(base_model, unfreeze: int):
    """
    Aquí va tu código para descongelar las unidades divisibles de la arquitectura efficient net
    https://keras.io/api/applications/efficientnet_v2/
    """
    pass
    
def unfreeze_residual_blocks(base_model, unfreeze: int, inference):
    # Encontrar todos los nombres de bloques únicos (formato convX_blockY)
    # Cada bloque tiene dentro batch norm, activación, convolución, etc, queremos descongelar todo eso de los últimos 'unfreeze' (n) bloques
    block_names = set()
    for layer in base_model.layers:
        if '_block' in layer.name:
            parts = layer.name.split('_')
            # El nombre del bloque está en las primeras dos partes (ejemplo: 'conv5', 'block3')
            if 'block' in parts[1]: 
                block_name = '_'.join(parts[:2])
                block_names.add(block_name)

    # Convertir a lista y ordenar
    block_names = sorted(list(block_names))
    
    print('All blocks', block_names)

    # Seleccionar los últimos 'n' bloques
    last_n_blocks = block_names[-unfreeze:]
    print('Unfreezed', last_n_blocks)

    # Descongelar las capas de los últimos 'n' bloques y congelar las de los demás
    for layer in base_model.layers:
        if any(block in layer.name for block in last_n_blocks): # Si pertenece a alguno de los bloques a descongelar
            if(inference and isinstance(layer, BatchNormalization)):
                layer.trainable = False
            else:
               layer.trainable = True
        else:
            layer.trainable = False

def prepare_training(optimizer, opt_hparams, l_rate, b_size, patience_lr, patience_es, metric, base_model, model):
    # Por el hecho de usar multistep, necesitamos reiniciar los elementos de entrenamiento del modelo en ambas fases instanciándolos de nuevo
    if(optimizer == 'Adam'):
        opt = Adam(learning_rate=l_rate, **opt_hparams)
    elif(optimizer == 'SGD'):
        opt = SGD(learning_rate=l_rate, **opt_hparams)
    elif optimizer =='AdamW':
        opt = AdamW(learning_rate=l_rate, **opt_hparams)
    
    preprocess_function = get_preprocessing(base_model)
    
    # Preparar los datos y los generadores según el batch size
    objs_train = [(ann.filename, obj) for ann in anns_train for obj in ann.objects]
    objs_valid = [(ann.filename, obj) for ann in anns_valid for obj in ann.objects]
    train_steps = math.ceil(len(objs_train)/b_size)
    valid_steps = math.ceil(len(objs_valid)/b_size)
    train_generator = generator_images(objs_train, b_size, 1, len(objs_train), preprocess_function, do_shuffle=True) # randomizar para no sesgar batches con orden
    valid_generator = generator_images(objs_valid, b_size, 1, len(objs_train), preprocess_function, do_shuffle=False) # no es necesario randomizar, solo se testean    
    
    callbacks, time_string = get_callbacks(patience_lr, patience_es)

    model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=metric)
    model.summary()
    
    return opt, train_generator, valid_generator, train_steps, valid_steps, callbacks, time_string

def transfer_train(l_rate, b_size, epochs, optimizer, opt_hparams, neurons_hidden_layers, activ_fun,
               patience_lr, patience_es, class_weight, metric, base_model = None, unfreeze = 0, multi_step= False, inference = False):
    # Congelamos todo el modelo
    base_model.trainable = False
    inputs = base_model.input
    x = base_model.output
    x = GlobalAveragePooling2D()(x) # Reshape
    
    for i in range(len(neurons_hidden_layers)): # Add ffNN to the top
        x = Dense(neurons_hidden_layers[i], activation=activ_fun)(x)
        #x = Dropout(0.5)(x)
        #x = BatchNormalization()(x)
    
    predictions = Dense(len(categories), activation='softmax')(x)

    model = Model(inputs=inputs, outputs=predictions)
    model._name = base_model.name # update name
    
    if multi_step:
        '''
        En este caso, el primer entrenamiento es con todo el modelo congelado y 
        solo ultimas capas por tanto el lr debe ser normal. 
        '''
        opt, train_generator, valid_generator, train_steps, valid_steps, callbacks, time_string = prepare_training(optimizer, opt_hparams, l_rate, b_size, patience_lr, patience_es, metric, base_model, model)
        print_params(time_string, l_rate, b_size, 1, epochs, optimizer, opt_hparams, 
                 neurons_hidden_layers, activ_fun, patience_lr, patience_es, unfreeze)
        
        h = model.fit(train_generator, validation_data=valid_generator, 
                  steps_per_epoch=train_steps, validation_steps=valid_steps, 
                  epochs=2*epochs, callbacks=callbacks, verbose=1, 
                  class_weight = class_weight)
        l_rate = 1e-5 # La segunda fase con learning rate muy pequeño
        
    '''
    Congelar las Capas del Modelo Preentrenado: todas menos las unfreeze últimas
    '''
    if not(multi_step and unfreeze == 0):
        unfreeze_model(base_model, unfreeze, inference) 
    
    opt, train_generator, valid_generator, train_steps, valid_steps, callbacks, time_string = prepare_training(optimizer, opt_hparams, l_rate, b_size, patience_lr, patience_es, metric, base_model, model)
    print_params(time_string, l_rate, b_size, 1, epochs, optimizer, opt_hparams, 
                 neurons_hidden_layers, activ_fun, patience_lr, patience_es, unfreeze)    

    h = model.fit(train_generator, validation_data=valid_generator, 
                  steps_per_epoch=train_steps, validation_steps=valid_steps, 
                  epochs=epochs, callbacks=callbacks, verbose=1, 
                  class_weight = class_weight)
    
    return model, h


In [ ]:
import math
from sklearn.metrics import confusion_matrix
from tensorflow.keras.saving import load_model

def show_test_results(model):
    preprocess_function = get_preprocessing(model)
    y_true, y_pred = test_net(model, anns_test, preprocess_function)
    # Compute the confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=list(categories.values()))
    draw_confusion_matrix(cm, categories)
    print_results(cm)

def training_loop(base_model):
    # Training loop
    for i in range(num_models):
        N_steps = (N_train/b_size[i]) * N_epochs[i]
        if isinstance(l_rate[i], list): # Si es cosine decay
            lr = get_lr_schedule(N_steps, l_rate[i][0], l_rate[i][1], l_rate[i][2], l_rate[i][3])
        else:
            lr = l_rate[i]

        
        model, h = transfer_train(lr, b_size[i], N_epochs[i], optimizer[i], opt_hparams[i], 
                                  neurons_hidden_layers[i], activ_fun[i],  
                                  patience_lr[i], patience_es[i],
                                  class_weight[i],metric[i], base_model, unfreeze[i], multi_step[i], inference[i])

        
        if type(metric[i][0]) != str:
            val_indexer = 'val_'+metric[i][0].name
        else:
            val_indexer = 'val_'+metric[i][0]

        best_idx = int(np.argmax(h.history[val_indexer]))
        best_value = np.max(h.history[val_indexer])
        print('Best validation model: epoch ' + str(best_idx+1), ' - '+val_indexer+': ' + str(best_value))
        show_test_results(model)
        

In [ ]:
from tensorflow.keras.applications.resnet_v2 import ResNet152V2

base_model = ResNet152V2(weights='imagenet', include_top=False)

In [ ]:
from tensorflow.keras.saving import load_model

process = 'transfer'

if(process == 'transfer'):
    training_loop(base_model)  
elif(process == 'LOAD'):
    model = load_model('/kaggle/working/2023-11-18_16-41-57.tf')
    model._name = base_model.name
    show_test_results(model)